# 06 · 端到端机器学习工作流 End-to-End ML Workflow

> **能力标签**：SH7（经典机器学习 / Classical ML）

## 目标 Objectives

完成本课后，你应该能够：

1. 使用 `sklearn.pipeline.Pipeline` 组合**预处理**（preprocessing）+ **模型**（model）防止数据泄露。
2. 在 `make_synth_table()` 合成表格数据上完成完整 ML 工作流：EDA → 特征工程 → 训练 → 交叉验证 → 评估。
3. 理解 `ColumnTransformer` 对不同类型特征（数值/类别）分别处理的方式。
4. 关联综合 W5 题包，综合运用 CV/LogReg/PCA/Metrics/XGBoost。

> 对应能力：**SH7**

## 概念讲解 Concepts

### 为何用 Pipeline？

若在训练前对**全部数据**（含测试集）做标准化，则测试集信息"泄露"到预处理参数（如均值/方差）中，导致评估过于乐观。

`Pipeline` 确保每个 CV 折只在**训练折**上 `fit` 变换器，在验证折上只 `transform`：

```
Pipeline([
    ("preprocessor", ColumnTransformer([...])),
    ("classifier",   XGBClassifier(...)),
])
```

---

### 典型工作流步骤

| 步骤 | 工具 |
|------|------|
| **EDA**：数据描述 | `df.describe()`, `value_counts()` |
| **特征工程**：缺失处理 | `SimpleImputer` |
| **特征工程**：数值标准化 | `StandardScaler` |
| **特征工程**：类别编码 | `OrdinalEncoder` / `OneHotEncoder` |
| **训练与 CV** | `Pipeline` + `StratifiedKFold` + `cross_val_score` |
| **评估** | `roc_auc_score`, `calibration_curve` |

---

### `make_synth_table` 数据集

合成表格数据，共 7 列：

| 列 | 类型 | 含义 |
|----|------|------|
| `record_id` | int | 记录 ID（不用于建模） |
| `age` | float | 年龄 |
| `group` | str | 类别 (A/B) |
| `x1` | float | 数值特征 1 |
| `x2` | float | 数值特征 2 |
| `x3` | float | 数值特征 3 |
| `label` | int | 二分类标签（0/1） |

## 示例 Worked Example — 端到端 Pipeline on `make_synth_table()`

In [ ]:
import sys, pathlib
_root = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (c / "projects" / "data").is_dir()), pathlib.Path.cwd())
sys.path.insert(0, str(_root))
from projects.data.make_synth_table import make_synth_table


In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

from projects.data.make_synth_table import make_synth_table

# ── 1. 加载数据 ───────────────────────────────────────────────────────────
df = make_synth_table(n=600, seed=0)
print("=== 数据概览 ===")
print(df.describe(include="all").T[["count", "mean", "std", "min", "max"]])
print(f"\n标签分布: {df['label'].value_counts().to_dict()}")

# ── 2. 特征 / 标签 ────────────────────────────────────────────────────────
feature_cols = ["age", "group", "x1", "x2", "x3"]
X = df[feature_cols]
y = df["label"].values

num_cols = ["age", "x1", "x2", "x3"]
cat_cols  = ["group"]

# ── 3. 构建 Pipeline ──────────────────────────────────────────────────────
num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
cat_transformer = Pipeline([
    ("imputer",  SimpleImputer(strategy="most_frequent")),
    ("encoder",  OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])
preprocessor = ColumnTransformer([
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols),
])

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=100, max_depth=3, tree_method="hist",
        eval_metric="logloss", random_state=0,
    )),
])

# ── 4. 5-Fold 交叉验证 ────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = cross_val_score(pipe, X, y, cv=skf, scoring="roc_auc")
print(f"\n5-Fold CV ROC-AUC: {cv_scores.round(3)}")
print(f"均值 ± 标准差: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# ── 5. 训练集 fit + 留出集评估（80/20）───────────────────────────────────
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            stratify=y, random_state=0)
pipe.fit(X_tr, y_tr)
prob_te = pipe.predict_proba(X_te)[:, 1]
auc_te  = roc_auc_score(y_te, prob_te)
print(f"\n留出集 ROC-AUC = {auc_te:.4f}")

# ── 6. 校准曲线可视化 ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
frac_pos, mean_pred = calibration_curve(y_te, prob_te, n_bins=8)
ax.plot(mean_pred, frac_pos, "s-", color="steelblue",
        label=f"XGBoost Pipeline (AUC={auc_te:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="完美校准")
ax.set_xlabel("平均预测概率")
ax.set_ylabel("实际阳性比例")
ax.set_title("校准曲线 — make_synth_table 数据集")
ax.legend()
fig.tight_layout()

_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "workflow_calibration.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("图像已保存到:", _outpath)
print("\n✓ Pipeline 防止数据泄露；CV 给出无偏泛化估计。")

## 动手 Exercise

In [ ]:
import sys, pathlib
_root = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (c / "projects" / "data").is_dir()), pathlib.Path.cwd())
sys.path.insert(0, str(_root))
from projects.data.make_synth_table import make_synth_table


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from projects.data.make_synth_table import make_synth_table

# 练习：用逻辑回归替换 XGBoost，比较 CV AUC
df = make_synth_table(n=600, seed=42)
feature_cols = ["age", "group", "x1", "x2", "x3"]
X = df[feature_cols]
y = df["label"].values
num_cols = ["age", "x1", "x2", "x3"]
cat_cols  = ["group"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scl", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), cat_cols),
])

pipe_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(C=1.0, max_iter=1000)),
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_lr = cross_val_score(pipe_lr, X, y, cv=skf, scoring="roc_auc")
print(f"LogReg Pipeline  CV AUC: {cv_lr.mean():.3f} ± {cv_lr.std():.3f}")
print("\n综合训练：一条 Pipeline 统一了预处理、模型与评估，确保无泄露。")

## 延伸阅读 Further Reading

1. **sklearn Pipeline + ColumnTransformer**：<https://scikit-learn.org/stable/modules/compose.html>
2. **XGBoost 文档**：<https://xgboost.readthedocs.io/en/stable/>
3. **《Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow》** Géron — 第 2 章（端到端工作流）
4. **Kaggle Learn — Intermediate ML**：<https://www.kaggle.com/learn/intermediate-machine-learning>（Pipeline & Cross-Validation）

## 关联练习 Related Assignments

综合 W5 题包（全部五个）：

```bash
python check.py 01-cv
python check.py 02-logreg
python check.py 03-pca
python check.py 04-metrics
python check.py 05-xgb
```

> 完成全部 W5 题包即达成能力标签 **SH7** · threshold ≥ 0.7